In [3]:
import h5py
import numpy as np

def wash_to_new_h5(
    in_path: str,
    out_path: str,
    data_key: str = "data",
    ch_start: int = 2000,
    ch_end: int = 5000,
    inclusive: bool = True,
    time_chunk: int = 64,
    out_dtype=np.float32,
    save_baseline: bool = True,
):
    """
    Build a new H5 file with the same structure as input, but with washed `data`.

    For each time t:
      A[t] = median(data[t, ch_start:ch_end])
      washed_data[t, :] = data[t, :] - A[t]
    """
    # Python slicing: left-closed right-open
    start = int(ch_start)
    stop = int(ch_end) + 1 if inclusive else int(ch_end)

    with h5py.File(in_path, "r") as f1, h5py.File(out_path, "w") as f2:
        # ---- 0) Copy file-level attributes (optional but helps keep "same format") ----
        for k, v in f1.attrs.items():
            f2.attrs[k] = v

        # ---- 1) Copy all top-level keys except `data` ----
        # This preserves your file structure (depth, stamps, any other datasets or groups).
        for key in f1.keys():
            if key == data_key:
                continue
            f1.copy(key, f2)

        # ---- 2) Create output dataset for washed data ----
        dset = f1[data_key]                 # original data: (time, channels)
        nt, nc = dset.shape

        create_kwargs = {}
        # try to mimic original compression/chunks if present
        if dset.compression is not None:
            create_kwargs["compression"] = dset.compression
            if dset.compression_opts is not None:
                create_kwargs["compression_opts"] = dset.compression_opts

        # use chunking friendly for time-wise streaming
        create_kwargs["chunks"] = (min(time_chunk, nt), nc)

        out = f2.create_dataset(
            data_key, shape=(nt, nc), dtype=out_dtype, **create_kwargs
        )

        # optional baseline dataset
        if save_baseline:
            base_key = f"baseline_median_ch_2{ch_start}_{ch_end}"
            baseline = f2.create_dataset(
                base_key, shape=(nt,), dtype=out_dtype, chunks=(min(time_chunk, nt),)
            )

        # ---- 3) Stream: compute A on the fly per chunk, subtract, write ----
        for i in range(0, nt, time_chunk):
            j = min(i + time_chunk, nt)

            sub = dset[i:j, start:stop]  # (chunk, selected_channels)
            A_block = np.median(sub, axis=1).astype(out_dtype)  # (chunk,)

            block = dset[i:j, :].astype(out_dtype)              # (chunk, nc)
            block -= A_block[:, None]
            out[i:j, :] = block

            if save_baseline:
                baseline[i:j] = A_block

    print("Saved washed H5 to:", out_path)


# ---------- usage ----------
# in_path = r"C:/Users/fengxiang.mao/Desktop/Devon data/HFTS-1_Devon/Evo_5_6_7_Sep_2025/Evo 7 (flowback)/Zgabay A14H - flowback - strain change.h5"
# out_path = r"C:/Users/fengxiang.mao/Desktop/Devon data/HFTS-1_Devon/Evo_5_6_7_Sep_2025/Evo 7 (flowback)/Strain_change_washed.h5"

# wash - 1
# in_path = r"C:/Users/fengxiang.mao/Desktop/Devon data/HFTS-1_Devon/Evo_5_6_7_Sep_2025/Evo 6 (choke)/Zgabay A14H - choke - strain change.h5"
# out_path = r"C:/Users/fengxiang.mao/Desktop/Devon data/HFTS-1_Devon/Evo_5_6_7_Sep_2025/Evo 6 (choke)/Strain_change_washed.h5"

# wash - 2 
in_path = r"C:/Users/fengxiang.mao/Desktop/Devon data/HFTS-1_Devon/Evo_5_6_7_Sep_2025/Evo 6 (choke)/Strain_change_washed.h5"
out_path = r"C:/Users/fengxiang.mao/Desktop/Devon data/HFTS-1_Devon/Evo_5_6_7_Sep_2025/Evo 6 (choke)/Strain_change_washed_2.h5"



wash_to_new_h5(in_path, out_path, ch_start=2000, ch_end=5000, inclusive=True, time_chunk=64)


Saved washed H5 to: C:/Users/fengxiang.mao/Desktop/Devon data/HFTS-1_Devon/Evo_5_6_7_Sep_2025/Evo 6 (choke)/Strain_change_washed_2.h5


In [4]:
import os
import h5py
import numpy as np

def build_rate_h5_from_washed(
    in_washed_path: str,
    out_rate_path: str,
    data_key: str = "data",
    time_chunk: int = 64,
    out_dtype=np.float32,
    overwrite: bool = False,
):
    """
    Create a new H5 with same structure as input washed H5, but `data` becomes
    strain-change-rate defined by:
      rate[0, :] = 0
      rate[t, :] = washed[t, :] - washed[t-1, :]   (Δt treated as 1)
    """
    if (not overwrite) and os.path.exists(out_rate_path):
        raise FileExistsError(f"Output file already exists: {out_rate_path}")

    with h5py.File(in_washed_path, "r") as f1, h5py.File(out_rate_path, "w") as f2:
        # Copy file-level attributes
        for k, v in f1.attrs.items():
            f2.attrs[k] = v

        # Copy everything except the main data_key; we will regenerate it
        for key in f1.keys():
            if key == data_key:
                continue
            f1.copy(key, f2)

        dset = f1[data_key]
        nt, nc = dset.shape

        # Mimic compression settings if any
        create_kwargs = {}
        if dset.compression is not None:
            create_kwargs["compression"] = dset.compression
            if dset.compression_opts is not None:
                create_kwargs["compression_opts"] = dset.compression_opts

        # Chunking: time-wise streaming
        create_kwargs["chunks"] = (min(time_chunk, nt), nc)

        out = f2.create_dataset(
            data_key,
            shape=(nt, nc),
            dtype=out_dtype,
            **create_kwargs
        )

        # Edge case: empty dataset
        if nt == 0:
            return

        prev_row = None

        for i in range(0, nt, time_chunk):
            j = min(i + time_chunk, nt)
            block = dset[i:j, :].astype(out_dtype)  # (j-i, nc)

            rate_block = np.empty_like(block)

            if i == 0:
                # First time sample -> all zeros
                rate_block[0, :] = 0
                if j - i > 1:
                    rate_block[1:, :] = block[1:, :] - block[:-1, :]
            else:
                # First row of this block uses previous row from last block
                rate_block[0, :] = block[0, :] - prev_row
                if j - i > 1:
                    rate_block[1:, :] = block[1:, :] - block[:-1, :]

            out[i:j, :] = rate_block
            prev_row = block[-1, :].copy()

    print("Saved rate H5 to:", out_rate_path)


# ----------------- usage -----------------
# in_washed = r"C:/Users/fengxiang.mao/Desktop/Devon data/HFTS-1_Devon/Evo_5_6_7_Sep_2025/Evo 7 (flowback)/Strain_change_washed.h5"
# out_rate  = r"C:/Users/fengxiang.mao/Desktop/Devon data/HFTS-1_Devon/Evo_5_6_7_Sep_2025/Evo 7 (flowback)/Strain_change_rate_washed.h5"

# wash - 1
# in_washed = r"C:/Users/fengxiang.mao/Desktop/Devon data/HFTS-1_Devon/Evo_5_6_7_Sep_2025/Evo 6 (choke)/Strain_change_washed.h5"
# out_rate  = r"C:/Users/fengxiang.mao/Desktop/Devon data/HFTS-1_Devon/Evo_5_6_7_Sep_2025/Evo 6 (choke)/Strain_change_rate_washed.h5"

# wash - 2
in_washed = r"C:/Users/fengxiang.mao/Desktop/Devon data/HFTS-1_Devon/Evo_5_6_7_Sep_2025/Evo 6 (choke)/Strain_change_washed_2.h5"
out_rate  = r"C:/Users/fengxiang.mao/Desktop/Devon data/HFTS-1_Devon/Evo_5_6_7_Sep_2025/Evo 6 (choke)/Strain_change_rate_washed_2.h5"


build_rate_h5_from_washed(in_washed, out_rate, time_chunk=64, overwrite=True)

# 之后读取（和你原代码完全一致，只是换文件路径）
f_rate = h5py.File(out_rate, "r")
dstrain_rate = f_rate["data"]
print(dstrain_rate.shape)  # (3415, 57204)


Saved rate H5 to: C:/Users/fengxiang.mao/Desktop/Devon data/HFTS-1_Devon/Evo_5_6_7_Sep_2025/Evo 6 (choke)/Strain_change_rate_washed_2.h5
(1458, 57168)


In [5]:
import numpy as np
import h5py

with h5py.File(in_washed, "r") as fw, h5py.File(out_rate, "r") as fr:
    X = fw["data"]
    R = fr["data"]

    # 第一行应为 0
    print("row0 max abs:", np.max(np.abs(R[0, :])))

    # 随便抽一个时间点比对：R[t] == X[t]-X[t-1]
    t = 100
    err = np.max(np.abs(R[t, :] - (X[t, :] - X[t-1, :])))
    print("check t=100 max abs err:", err)


row0 max abs: 0.0
check t=100 max abs err: 0.0


In [6]:
import numpy as np
import h5py

with h5py.File(out_rate, "r") as fr:
    R = fr["data"]
    print("any nan:", np.isnan(R[:100, :]).any())
    print("any inf:", np.isinf(R[:100, :]).any())
    print("mean (sample):", np.mean(R[:200, :]))
    print("std  (sample):", np.std(R[:200, :]))


any nan: False
any inf: False
mean (sample): 0.00290858
std  (sample): 47.091877
